# Plotting from `n.statistics`

A guided ~40-min follow-up to the statistics session. Every `n.statistics.<method>`
exposes two plotting accessors:

- `.plot`  — matplotlib/seaborn, static
- `.iplot` — plotly, interactive

All the familiar statistics filters (`carrier`, `bus_carrier`, `nice_names`) pass
straight through. Layout (`x`, `y`, `color`, `facet_col`, `facet_row`) replaces the
`groupby*` knobs from the raw API.

We reuse `pypsa.examples.carbon_management()` from the previous session.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly.io as pio
import pypsa

import _patches  # noqa: F401 — monkey-patches plotly categorical y-axis bug

_patches.suppress_pypsa_copy_warning()

mpl.rcdefaults()
pio.renderers.default = "notebook_connected"

n = pypsa.examples.carbon_management()
n.carriers["nice_name"] = n.carriers["nice_name"].str.replace(r"\$_2\$", "₂", regex=True)
s = n.statistics

:::{tip}
Same `s = n.statistics` shorthand from last session — now `s.capex.plot.bar(...)` is the pattern.
:::

# Why use the accessors?

Last session ended with `eb.xs(...).droplevel(0).sort_values().plot.barh(...)`. That works,
but you carry reshape logic yourself. The accessors handle filtering, layout, units, nice
names, and pick carrier colors from `n.carriers.color` automatically.

In [ ]:
# manual way (recap from last session) — energy_balance is indexed by
# (component, bus_carrier, carrier); xs drops bus_carrier, droplevel drops component
eb = s.energy_balance()
eb.xs("AC", level="bus_carrier").droplevel("component").sort_values().plot.barh(figsize=(8, 5), xlabel="MWh")

In [ ]:
# accessor way — same result, one line, carrier colors applied
s.energy_balance.plot.bar(bus_carrier="AC", aspect=2)

:::{note}
The accessor is called on the **method itself**, not on its return value:
`s.capex.plot.bar()` ✅ — `s.capex().plot.bar()` is plain pandas and loses the schema/colors.
:::

:::{important}
- `.plot.*` returns `(fig, ax, FacetGrid)` — except `.plot.map`, which returns `(fig, ax)`.
- `.iplot.*` returns a `plotly.graph_objects.Figure`.

Unpack as `fig, ax, g = s.capex.plot.bar()` if you want to keep editing the axes.
:::

:::{warning}
Three statistics kwargs are **reserved** and raise `ValueError` if passed:

- `groupby` — derived from `x` / `y` / `color` / `facet_*`
- `groupby_time` — set to `"sum"` unless `snapshot` appears in a layout kwarg
- `aggregate_across_components` — set to `True` unless `component` appears in a layout kwarg

For a time series, set `x="snapshot"`. To split by component, set `color="component"`.
:::

# `.plot` vs `.iplot`

| Accessor | Backend | Use for | Plot types |
|---|---|---|---|
| `.plot` | matplotlib/seaborn | reports, papers, slides, PNG/PDF | `bar`, `line`, `area`, `scatter`, `box`, `violin`, `histogram`, `map` |
| `.iplot` | plotly | exploration, hover, zoom, HTML export | `bar`, `line`, `area` only |

:::{warning}
`.iplot.scatter`, `.iplot.box`, `.iplot.violin`, `.iplot.histogram`, `.iplot.map`
all raise. Plotly coverage in the accessor is limited to `bar`/`line`/`area`. For
distributions and maps, use `.plot.*`.
:::

:::{tip}
The examples below mostly use `.iplot` for interactivity. Each cell includes a
commented `.plot` equivalent — uncomment to try the static version.
:::

In [ ]:
s.capex.plot.bar(carrier=["solar", "onwind", "offwind"], aspect=2)

In [ ]:
s.capex.iplot.bar(carrier=["solar", "onwind", "offwind"])

## Carrier colors

When `color in (None, "carrier")` the palette comes from `n.carriers.color`. In
plotly the same colors become `color_discrete_map`. Set `nice_names=False` for
raw carrier identifiers.

In [ ]:
n.carriers[["nice_name", "color"]].head()

# Plot types

One representative carbon_management example per plot type. Each call pulls a
different statistics method and exercises a different filter / layout kwarg.

## `bar` — categorical totals

The workhorse. Default is horizontal bars (`x="value"`, `y="carrier"`), stacked
by carrier color.

In [ ]:
s.capex.iplot.bar()
# s.capex.plot.bar(aspect=2, height=7)

In [ ]:
# bus_carrier filter; grouped bars instead of stacked
s.capex.iplot.bar(bus_carrier="AC")
# s.capex.plot.bar(bus_carrier="AC", stacked=False, aspect=2, height=6)

In [ ]:
fig = s.optimal_capacity.iplot.bar(bus_carrier=["AC", "low voltage"], facet_col="bus_carrier")
fig.update_layout(showlegend=False)
# s.optimal_capacity.plot.bar(bus_carrier=["AC", "low voltage"], facet_col="bus_carrier", height=6)

In [ ]:
countries = ["DE", "FR", "IT"]
query = f"country in {countries}"

s.supply.iplot.bar(x="bus_carrier", y="value", color="carrier", facet_row="country", sharey=False, query=query)
# s.supply.plot.bar(x="bus_carrier", y="value", color="carrier", facet_row="country", sharey=False, query=query, aspect=2)

In [ ]:
s.capex.iplot.bar(stacked=True, y="country", bus_carrier="gas")

In [ ]:
# same plot, unstacked and filtered to DE/FR/IT, with non-zero values only
s.capex.iplot.bar(stacked=False, y="country", bus_carrier="gas", height=500, query=f"{query} and value > 0")

:::{tip}
`query="value > 0"` is a handy pandas-style filter on the long-format frame —
useful for dropping zeros from a busy bar chart.
:::

## `line` — time series

For time-resolved statistics (`energy_balance`, `supply`, `withdrawal`, `opex`, …)
the schema already defaults `x="snapshot"`, so a plain call returns a snapshot-wise line.

In [ ]:
s.energy_balance.iplot.line(bus_carrier="AC", carrier=["solar", "onwind", "nuclear"], color="carrier")
# s.energy_balance.plot.line(bus_carrier="AC", carrier=["solar", "onwind", "nuclear"], color="carrier")

In [ ]:
# explicit x="snapshot" + color by carrier
s.supply.iplot.line(bus_carrier=["AC", "low voltage"], x="snapshot", color="carrier", facet_col="bus_carrier", facet_row="country", query=query)

:::{note}
To go back to a non-time-series view, override `x` to a categorical column (e.g. `x="carrier"`).
:::

## `area` — stacked dispatch

The classic stacked dispatch plot in one call. `energy_balance` defaults to
`kind="area"`, so `.plot()` alone works.

In [ ]:
s.energy_balance.iplot.area(bus_carrier="AC")
# s.energy_balance.plot.area(bus_carrier="AC")

In [ ]:
# H2 vs AC sector balance, faceted by bus_carrier
s.energy_balance.iplot.area(bus_carrier=["H2", "AC"], facet_row="bus_carrier")
# s.energy_balance.plot.area(bus_carrier=["H2", "AC"], facet_row="bus_carrier")

:::{tip}
Mixed-sign areas (positive supply / negative withdrawal) are split by sign and
stacked separately — no extra config needed.
:::

## Gaining More Control

### Parameter pass-through recap

Every plot type accepts the statistics filter kwargs:

In [ ]:
s.supply.iplot.bar(
    bus_carrier="AC",
    carrier=["solar", "onwind", "OCGT"],
    nice_names=True,
)

:::{note}
The reserved-kwargs rule (`groupby`, `groupby_time`, `aggregate_across_components`) was
called out at the top of this notebook. If you forget, the error message is helpful.
:::

### Extra kwargs for sizing, faceting & styling

Beyond the statistics filters, both backends accept layout kwargs. Extra `**kwargs`
flow straight to the underlying seaborn / plotly.express call.

**Shared kwargs** (both `.plot` and `.iplot`):

| kwarg | effect |
|---|---|
| `x`, `y`, `color` | pick which long-format column maps to each axis / hue |
| `facet_col`, `facet_row` | split into subplots by a column |
| `stacked` | stack bars/areas (`True`, default) or group them (`False`) |
| `query` | pandas query string applied to the long-format frame, e.g. `"value > 0"` |
| `sharex`, `sharey` | share axis limits across facets (default `True` when the axis is `"value"`) |

**`.plot` only** (seaborn `FacetGrid`):

| kwarg | effect | default |
|---|---|---|
| `height` | height of each facet in inches | `3` (`4` for box/violin/histogram) |
| `aspect` | width = aspect × height | `2` for line/area, `1` otherwise |
| `despine` | remove top/right spines | `True` |
| `subplot_kws` | dict passed to `matplotlib.figure.Figure.add_subplot` | — |

**`.iplot` only** (plotly express):

| kwarg | effect | default |
|---|---|---|
| `height` | figure height in px | `500` |
| `width` | figure width in px | `800` |
| `title` | figure title | stat name, titlecased |
| `facet_col_wrap` | max columns before wrapping facets | — |
| `range_x`, `range_y` | explicit axis limits as `[min, max]` | — |
| `labels` | dict to rename axis / legend labels | — |

### Drop down to the underlying figure/axes

In [ ]:
from pathlib import Path

out = Path("../dev")
out.mkdir(exist_ok=True)

fig, ax, g = s.capex.plot.bar(height=6)
ax.set_title("CAPEX by carrier")
fig.savefig(out / "capex.png", dpi=150)

In [ ]:
fig = s.energy_balance.iplot.area(bus_carrier="AC")
fig.update_layout(title="AC dispatch", height=500)

# Spatial view — `.plot.map`

The headline of the session. `.plot.map` draws bus-level statistics as scaled
circles on the network and branch-level statistics (or transmission flows) as
line widths / arrows. Built from any statistics method except `prices`.

In [ ]:
# enlarge the default figure for the map cells below
# (re-run cell 1 — `mpl.rcdefaults()` — to revert if you rerun earlier cells)
plt.rcParams["figure.figsize"] = (12, 8)

:::{warning}
There is **no `.iplot.map`** in PyPSA 1.2.1 — `StatisticInteractivePlotter` doesn't
expose it. For interactive geographic plots, fall back to `n.iplot()` directly.
:::

:::{important}
`prices.plot.map` raises `NotImplementedError`. Every other statistic supports the map.
:::

In [ ]:
# optimal capacity on the AC layer
s.optimal_capacity.plot.map(bus_carrier="AC", title="Optimal capacity (AC)", branch_area_fraction=0, draw_legend_lines=False)

In [ ]:
fig, ax = s.optimal_capacity.plot.map(bus_carrier="AC", title="Optimal capacity (AC)", branch_area_fraction=0.00, draw_legend_lines=False)
s.transmission.plot.map(bus_carrier="AC", ax=ax)

In [ ]:
s.optimal_capacity.plot.map(bus_carrier="H2", title="Optimal capacity (H2)", branch_area_fraction=0.005)

In [ ]:
# capex on the map
fig, ax = s.capex.plot.map(bus_carrier="AC")

In [ ]:
# energy balance — split half-circles + flow arrows (schema defaults)
fig, ax = s.energy_balance.plot.map(bus_carrier="AC")

In [ ]:
# H2 network — sector coupling highlight
fig, ax = s.energy_balance.plot.map(bus_carrier="H2")

In [ ]:
# supply with branch widths instead of flow arrows
fig, ax = s.supply.plot.map(
    bus_carrier="AC",
    transmission_flow=False,
    draw_legend_arrows=False,
    draw_legend_lines=True,
)

In [ ]:
# storage capacity (allowed only for optimal_capacity / installed_capacity)
s.optimal_capacity.plot.map(bus_carrier="AC", storage=True)

In [ ]:
# customise legends
fig, ax = s.capex.plot.map(
    bus_carrier="AC",
    legend_circles_kw={"bbox_to_anchor": (0, 1.0), "title": "GW"},
    legend_patches_kw={"ncol": 2, "bbox_to_anchor": (1, 1)},
)

:::{note} Map kwargs at a glance
- `bus_carrier` — pick one (required in sector-coupled networks)
- `carrier` — filter components
- `transmission_flow` — branch widths (`False`) vs flow arrows (`True`). Default
  is `True` for `supply` / `withdrawal` / `energy_balance`, `False` elsewhere.
- `bus_split_circle` — pos/neg split halves; default `True` for `energy_balance`.
- `bus_area_fraction` / `branch_area_fraction` / `flow_area_fraction` —
  auto-scaling targets (fraction of plot area). Default `0.02`.
- `geomap` / `geomap_resolution` / `geomap_color` / `projection` / `boundaries`
  — basemap controls.
- `legend_circles_kw` / `legend_lines_kw` / `legend_arrows_kw` / `legend_patches_kw`
  — dict of matplotlib legend kwargs each.
:::

# Recap

| Plot type | `.plot` | `.iplot` | Best for |
|---|---|---|---|
| `bar` | ✓ | ✓ | totals by carrier / region |
| `line` | ✓ | ✓ | dispatch / price curves |
| `area` | ✓ | ✓ | stacked dispatch |
| `map` | ✓ | ✗ | spatial / network view |

`.plot.*` additionally exposes `scatter`, `box`, `violin`, `histogram` — not covered
here; see the pypsa docs.

:::{tip} Tips and tricks
- Call the accessor on the **method**, not its result: `s.capex.plot.bar()` ✅.
- Filter with `carrier=` / `bus_carrier=` / `query=`.
- Reshape with `x` / `y` / `color` / `facet_col` / `facet_row` — these replace
  the `groupby*` knobs.
- Never pass `groupby`, `groupby_time`, `aggregate_across_components` — they're derived.
- `.iplot` is limited to `bar`/`line`/`area`. For everything else, use `.plot`.
- `.iplot.map` does not exist — only `.plot.map`.
- For sector-coupled networks, **always** set `bus_carrier`.
- `.plot.*` returns `(fig, ax, FacetGrid)` (or `(fig, ax)` for `map`); `.iplot.*` returns a plotly `Figure`.
- Colors come from `n.carriers.color` — set them once, get consistent figures everywhere.
:::

:::{seealso}
[pypsa.org/latest/user-guide/statistics](https://docs.pypsa.org/latest/user-guide/statistics/) — plotting section.
:::